> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB10 v1.2 — PE Bayesian population model: R and total carbenicillin-resistant burden

**Purpose.** Exploratory comparison of **R** (recipients without Transmid), **T1** (resistant Transmid carriers), **T2** (all Transmid carriers), and **R + T1** (all carbenicillin-resistant bacteria) between target and non-target peritoneal (PE) groups. Both estimated CFU/mL and population-normalized outcomes are retained. This is *not* the conditional resistance-loss outcome assessed by NB05, which uses T1/T2 among Transmid-containing bacteria. The main outputs include **both absolute CFU/mL and fractions**, with separate comparisons at 6 h and at the non-time-matched later endpoints.

**Biological definitions.** There are no donors in PE. R is Transmid-negative recipient bacteria assumed to retain carbenicillin resistance. T1 is Transmid-positive, carbenicillin-resistant bacteria. T-sensitive = T2 − T1 is Transmid-positive, carbenicillin-sensitive bacteria. T2 = T1 + T-sensitive. We **assume spontaneous resistance-plasmid loss in R is zero** for this exploratory calculation; it has not been established experimentally.

**Plate-to-population mappings:** Cb = R + T1; Cb+Tc = T1; Tc = T1 + T-sensitive. The later animals also have nonselective LB counts, used solely as an external descriptive check. Original plate CFU/mL estimates are used without clipping or alteration; their inconsistencies are retained. *T-sensitive is a culture-defined phenotype, not proof of CRISPR gene cleavage.*

**Inputs.** 32 independent terminal PE animals (eight per group): target and non-target at 6 h; target at 16–20 h (one at 16 h, seven at 20 h), non-target at 24 h. The later contrast is **not time-matched**. Original KWS-derived PE measurements must be supplied separately with authorisation; no animal-level data are embedded.

**Population normalization (v1.2).** For each animal and each Bayesian posterior draw, the denominator is the estimated total **R + T2**, where **T2 = T1 + T-sensitive**. Report `f_R = R/(R+T2)`, `f_T1 = T1/(R+T2)` and `f_T2 = T2/(R+T2)` for target and non-target animals. **T1 is contained in T2**, so `f_R + f_T2 = 1`, **not** `f_R + f_T1 + f_T2 = 1`. The resistant fraction is `f_R + f_T1`; its complement is `(T2-T1)/(R+T2)`. This normalization controls for total estimated within-animal bacterial burden **but does not adjust for initial/baseline burden** or establish a causal treatment effect. Group fractions are summarized from per-animal posterior fractions, not obtained by dividing group median counts. v1.2 does not change the Stan likelihood or the original data.

**Changes from NB10 v1.0.** The earlier model failed convergence (R-hat 1.0382; minimum bulk ESS 85), mainly in weakly identified plate-error parameters. v1.1 uses a **simpler hierarchical latent log-population model** with *prespecified* log-scale plate measurement errors and animal heterogeneity, rather than estimating those noise parameters from only three selective plates per animal. These assumptions are **not empirically validated** and require sensitivity checks. Convergence can only be assessed after Stan sampling.

**Model.** Group-specific log means for each nonnegative latent population R, T1 and T-sensitive; animal deviations use fixed log-scale SDs. Observed log(CFU/mL) use Student-t(ν=4) likelihoods centred on the appropriate **sums** of latent populations. Positive latent populations imply consistent biological subset constraints, while observed plate counts remain free to disagree. LB measurements do not enter the fit.

**Outputs.** Normalized R, T1 and T2 tables and target/non-target contrasts are included. Under `06_Results/NB10_PE_v1_2/`: convergence, group and animal population summaries, target/non-target contrasts, 6-hour results flagged as time-matched, later exploratory contrasts, posterior predictive diagnostics and figures. These results do not modify NB05, NB09, NB10 v1.0, manuscripts, supplements or GitHub.

**Warning.** Differences between target and non-target groups in bacterial loads can predate treatment or reflect strain biology. A between-group difference does **not** establish a reduction caused by targeted CRISPR. Likewise, posterior probabilities are conditional on the stated model and fixed measurement-error assumptions. These are private KWS-derived data; redistribution rights remain unverified.

**Instructions.** Run cells sequentially. Original animal-level inputs must be separately authorised and supplied; they are not included in this release.


In [ ]:
#@title NB10.01 — Install analysis packages
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'cmdstanpy', 'numpy', 'pandas', 'matplotlib'])
print('Python analysis packages available.')


In [ ]:
#@title NB10.02 — Set up storage (Google Drive or local)
from pathlib import Path
PROJECT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    mydrive = Path('/content/drive/MyDrive')
    direct = mydrive / 'Transmid_Animal_Bayesian_Project'
    if direct.is_dir():
        PROJECT = direct
    else:
        matches = [p for p in mydrive.rglob('Transmid_Animal_Bayesian_Project') if p.is_dir()]
        if len(matches) == 1:
            PROJECT = matches[0]
        elif len(matches) > 1:
            raise RuntimeError(f'Multiple candidate project folders: {matches}')
except (ImportError, OSError) as exc:
    print('Google Drive not used:', exc)
if PROJECT is None:
    PROJECT = Path('/content/NB10_PE_Project')
    print('Existing project folder not found; outputs will be saved locally.')

DATA_DIR = PROJECT / '03_Analysis_Data' / 'NB10_PE'
MODEL_DIR = PROJECT / '05_Stan_Models'
RESULT_DIR = PROJECT / '06_Results' / 'NB10_PE_v1_2'
for p in (DATA_DIR, MODEL_DIR, RESULT_DIR): p.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)
print('Results:', RESULT_DIR)


In [ ]:
#@title NB10.03 / NB11.03 — Load separately authorised PE culture-count data
# Public-release version: original individual-animal observations are NOT included.
# Place an authorised copy of NB10_PE_Original_CFU_derived.csv in DATA_DIR
# after obtaining the applicable data-sharing permission.
import pandas as pd, numpy as np
from IPython.display import display
CSV_NAME = 'NB10_PE_Original_CFU_derived.csv'
INPUT = DATA_DIR / CSV_NAME
if not INPUT.is_file():
    raise FileNotFoundError(f'Private PE input not provided: {INPUT}. See README for required columns.')
data = pd.read_csv(INPUT, dtype={'animal_id':str})
GROUPS = ['Early — Target', 'Early — Non-target',
          'Later — Target', 'Later — Non-target']
COLS = ['Cb_CFU_ml', 'CbTc_CFU_ml', 'Tc_CFU_ml']
required = {'animal_id','group','hours_post_infection','LB_CFU_ml',*COLS}
assert required.issubset(data.columns), f'Missing PE columns: {required-set(data.columns)}'
assert len(data) == 32 and data.animal_id.nunique() == 32
assert set(data.group) == set(GROUPS)
assert all((data.group == g).sum() == 8 for g in GROUPS)
assert np.isfinite(data[COLS].values.astype(float)).all()
assert (data[COLS] > 0).all().all()
assert data[data.group.eq('Early — Target')].hours_post_infection.eq(6).all()
assert data[data.group.eq('Early — Non-target')].hours_post_infection.eq(6).all()
assert data[data.group.eq('Later — Target')].hours_post_infection.value_counts().to_dict() == {20:7,16:1}
assert data[data.group.eq('Later — Non-target')].hours_post_infection.eq(24).all()
assert data[data.group.str.startswith('Early')].LB_CFU_ml.isna().all()
assert data[data.group.str.startswith('Later')].LB_CFU_ml.notna().all()
print('PE input validated; 32 animals (private source supplied separately).')


In [ ]:
#@title NB10.04 — Audit plate-count inconsistencies without correction
audit = data[['animal_id','group','hours_post_infection',*COLS,'LB_CFU_ml']].copy()
audit['observed_R_naive'] = audit.Cb_CFU_ml - audit.CbTc_CFU_ml
audit['observed_T_sensitive_naive'] = audit.Tc_CFU_ml - audit.CbTc_CFU_ml
audit['CbTc_greater_than_Tc'] = audit.CbTc_CFU_ml > audit.Tc_CFU_ml
audit['CbTc_greater_than_Cb'] = audit.CbTc_CFU_ml > audit.Cb_CFU_ml
audit['Cb_greater_than_LB'] = audit.Cb_CFU_ml > audit.LB_CFU_ml
audit['Tc_greater_than_LB'] = audit.Tc_CFU_ml > audit.LB_CFU_ml
for col in ('CbTc_greater_than_Tc','CbTc_greater_than_Cb','Cb_greater_than_LB','Tc_greater_than_LB'):
    print(f'{col}: {int(audit[col].sum())} animals')
print('Naive differences are audit values only; negatives are NOT set to zero.')
display(audit[audit[['CbTc_greater_than_Tc','CbTc_greater_than_Cb',
                     'Cb_greater_than_LB','Tc_greater_than_LB']].any(axis=1)])
audit.to_csv(RESULT_DIR/'plate_count_audit.csv', index=False)


In [ ]:
#@title NB10.05 — Compile simplified latent CFU/mL Stan model
from cmdstanpy import CmdStanModel, cmdstan_path, install_cmdstan
import tempfile
try:
    print('CmdStan:',cmdstan_path())
except ValueError:
    print('Installing CmdStan in this runtime (once).')
    if not install_cmdstan(cores=2,progress=True):
        raise RuntimeError('CmdStan installation failed')

# Fixed observation uncertainties (natural-log units); these are working assumptions,
# not calibrated experimental measurement errors. Do not silently tune them to a desired result.
PLATE_LOG_SD = np.array([0.7, 0.7, 0.7], dtype=float)  # Cb, Cb+Tc, Tc
# Fixed between-animal heterogeneity for R, T1 and T-sensitive (natural-log units).
ANIMAL_LOG_SD = np.array([1.8, 1.8, 1.8], dtype=float)
assert np.all(PLATE_LOG_SD>0) and np.all(ANIMAL_LOG_SD>0)

STAN_CODE = r'''// NB10 v1.2: nonnegative R, T1, T-sensitive abundances; no donors in PE.
// Data are workbook CFU/mL estimates, not raw colony counts.
data {
  int<lower=1> N;
  int<lower=2> G;
  array[N] int<lower=1,upper=G> group;
  matrix<lower=0>[N,3] y;  // Cb, Cb+Tc, Tc
  vector<lower=0>[3] sd_plate;   // fixed, not estimated
  vector<lower=0>[3] sd_animal;  // fixed, not estimated
}
parameters {
  matrix[G,3] mu_log_population;  // per-group log CFU/mL means of R,T1,Ts
  matrix[N,3] z;                  // noncentred animal deviations
}
transformed parameters {
  matrix[N,3] log_population;
  for (i in 1:N)
    for (k in 1:3)
      log_population[i,k] = mu_log_population[group[i],k] + sd_animal[k]*z[i,k];
}
model {
  to_vector(mu_log_population) ~ normal(15,5);  // weakly informative proper prior
  to_vector(z) ~ std_normal();
  for (i in 1:N) {
    log(y[i,1]) ~ student_t(4,
       log_sum_exp(log_population[i,1], log_population[i,2]), sd_plate[1]);
    log(y[i,2]) ~ student_t(4, log_population[i,2], sd_plate[2]);
    log(y[i,3]) ~ student_t(4,
       log_sum_exp(log_population[i,2], log_population[i,3]), sd_plate[3]);
  }
}
generated quantities {
  matrix[N,3] log_y_rep;
  for (i in 1:N) {
    log_y_rep[i,1] = student_t_rng(4,
       log_sum_exp(log_population[i,1],log_population[i,2]), sd_plate[1]);
    log_y_rep[i,2] = student_t_rng(4,log_population[i,2],sd_plate[2]);
    log_y_rep[i,3] = student_t_rng(4,
       log_sum_exp(log_population[i,2],log_population[i,3]),sd_plate[3]);
  }
}
'''
MODEL_SOURCE = MODEL_DIR / 'NB10_v1_2_PE_R_T1_Tsensitive.stan'
if MODEL_SOURCE.exists() and MODEL_SOURCE.read_text()!=STAN_CODE:
    raise ValueError('A different v1.1 Stan source already exists. Nothing overwritten.')
if not MODEL_SOURCE.exists(): MODEL_SOURCE.write_text(STAN_CODE)
RUN_DIR = Path(tempfile.mkdtemp(prefix='NB10_v11_PE_'))
STAN_LOCAL = RUN_DIR / MODEL_SOURCE.name
STAN_LOCAL.write_text(STAN_CODE)
model = CmdStanModel(stan_file=str(STAN_LOCAL))
print('Model compiled:',STAN_LOCAL)
print('Input plate SD:',PLATE_LOG_SD,'; animal variation SD:',ANIMAL_LOG_SD)


In [ ]:
#@title NB10.06 — Fit revised PE model once
# Separate results folder preserves the original NB10 chain and output files.
group = [GROUPS.index(g)+1 for g in data.group]
y = data[COLS].to_numpy(dtype=float)
stan_input = {'N':len(data),'G':len(GROUPS),'group':group,'y':y,
              'sd_plate':PLATE_LOG_SD.tolist(),'sd_animal':ANIMAL_LOG_SD.tolist()}
CHAIN_DIR = RUN_DIR / 'chains'
CHAIN_DIR.mkdir(parents=True,exist_ok=True)
fit = model.sample(data=stan_input, chains=4, parallel_chains=2,
                   iter_warmup=1500, iter_sampling=1500,
                   seed=20261125, adapt_delta=0.95, max_treedepth=12,
                   output_dir=str(CHAIN_DIR), show_progress=True)
print('Sampling completed. Convergence must be checked in NB10.07.')


In [ ]:
#@title NB10.07 — Convergence diagnostics and validation gate
summary = fit.summary()
idx = summary.index.to_series().str.startswith((
    'mu_log_population[','z[','log_population['))
pars = summary.loc[idx].copy()
max_rhat = float(pars.R_hat.max())
min_ess = float(pars.ESS_bulk.min())
method_vars = fit.method_variables()
div = int(np.sum(method_vars['divergent__']))
trees = method_vars['treedepth__']
tree_hits = int(np.sum(trees>=12))
valid = (np.isfinite(max_rhat) and np.isfinite(min_ess) and
         max_rhat<1.01 and min_ess>=400 and div==0 and tree_hits==0)
print(f'max R-hat={max_rhat:.4f}, min ESS_bulk={min_ess:.0f}, '
      f'divergences={div}, treedepth hits={tree_hits}/{trees.size}')
print('CONVERGENCE:','PASS (not full biological validation)' if valid
      else 'FAIL — do not interpret as reliable posterior results')
pars[['Mean','StdDev','ESS_bulk','R_hat']].to_csv(
    RESULT_DIR/'stan_parameter_diagnostics.csv')
(RESULT_DIR/'model_validation.txt').write_text(
    f'NB10 v1.2 exploratory PE population model\nConvergence PASS: {valid}\n'
    f'Max R-hat: {max_rhat:.6f}\nMin ESS bulk: {min_ess:.0f}\n'
    f'Divergences: {div}\nTreedepth hits: {tree_hits}/{trees.size}\n'
    f'Fixed plate log SD: {PLATE_LOG_SD.tolist()}\n'
    f'Fixed animal log SD: {ANIMAL_LOG_SD.tolist()}\n'
    'Fixed error/heterogeneity values are unvalidated working assumptions.\n'
    'Later groups are not time matched; absolute loads are not baseline-adjusted.\n')
display(pars[['Mean','StdDev','ESS_bulk','R_hat']].sort_values('R_hat',ascending=False).head(18))


In [ ]:
#@title NB10.08 — Estimate group and individual populations and fractions
# CFU/mL are fitted latent populations; fractions refer to all viable modelled bacteria.
log_pop = np.asarray(fit.stan_variable('log_population'))
assert log_pop.shape[1:] == (len(data),3)
pop = np.exp(log_pop)  # R, T1, T-sensitive, in CFU/mL
R,T1,Ts = (pop[:,:,k] for k in range(3))
T2 = T1 + Ts
res_burden = R + T1
all_burden = R + T2
values = {
 'R_CFU_ml':R, 'T1_CFU_ml':T1, 'T_sensitive_CFU_ml':Ts,
 'T2_CFU_ml':T2, 'resistant_R_plus_T1_CFU_ml':res_burden,
 'total_R_plus_T2_CFU_ml':all_burden,
 'R_fraction':R/all_burden,
 'T1_fraction':T1/all_burden,
 'resistant_fraction':res_burden/all_burden,
 'T2_fraction':T2/all_burden,
 'sensitive_T_fraction':Ts/all_burden,
 'sensitive_within_T2':Ts/T2,
}
# T1 is a subset of T2, and only R+T2 forms a complete partition.
assert np.allclose(values['R_fraction']+values['T2_fraction'], 1.0, atol=1e-10)
assert np.all(values['T1_fraction']<=values['T2_fraction']+1e-10)
assert np.allclose(values['R_fraction']+values['T1_fraction'],
                   values['resistant_fraction'], atol=1e-10)
def quants(arr):
    q = np.quantile(arr,[0.025,0.5,0.975]); return [float(v) for v in q]
rows=[]; animal_rows=[]
for g in GROUPS:
    m=(data.group==g).to_numpy()
    for label,arr in values.items():
        # Compare both group arithmetic averages and within-group median across animals.
        for agg,group_draw in [('mean',arr[:,m].mean(axis=1)),
                               ('median_across_animals',np.median(arr[:,m],axis=1))]:
            lo,mid,hi=quants(group_draw)
            rows.append(dict(group=g,quantity=label,aggregation=agg,
                             q025=lo,median=mid,q975=hi))
for i,animal in data.iterrows():
    for label,arr in values.items():
        lo,mid,hi=quants(arr[:,i])
        animal_rows.append(dict(animal_id=animal.animal_id,group=animal.group,
            hours_post_infection=animal.hours_post_infection,
            quantity=label,q025=lo,median=mid,q975=hi))
by_group = pd.DataFrame(rows)
by_animal = pd.DataFrame(animal_rows)
by_group.to_csv(RESULT_DIR/'posterior_group_population_and_fraction.csv',index=False)
by_animal.to_csv(RESULT_DIR/'posterior_animal_population_and_fraction.csv',index=False)
print('Primary outcomes — posterior group means across eight study animals:')
focus = ['R_CFU_ml','T1_CFU_ml','T2_CFU_ml',
         'resistant_R_plus_T1_CFU_ml','R_fraction','T1_fraction',
         'T2_fraction','resistant_fraction']
display(by_group[(by_group.aggregation=='mean') & by_group.quantity.isin(focus)].round(5))
# Focused tables: one value per group x component, with posterior 95% CrIs.
normalized_names = ['R_fraction','T1_fraction','T2_fraction']
normalized_group = by_group[(by_group.aggregation=='mean') &
                            by_group.quantity.isin(normalized_names)].copy()
normalized_group['denominator']='estimated R + T2 for each animal'
normalized_group.to_csv(RESULT_DIR/'normalized_R_T1_T2_by_group.csv',index=False)
print('Normalized R, T1 and T2 (posterior mean across animals, with 95% credible intervals):')
display(normalized_group.round(5))
if not valid:print('NOT VALIDATED: posterior summaries provisional because convergence failed.')


In [ ]:
#@title NB10.09 — Compare target versus non-target: R, T1, T2 and resistant burden
# Differences represent group contrasts, not causal effects or changes from baseline.
comparisons=[]
for endpoint,gt,gn in [
    ('Early 6 h — time-matched','Early — Target','Early — Non-target'),
    ('Later — NOT time-matched','Later — Target','Later — Non-target')]:
    mt=(data.group==gt).to_numpy(); mn=(data.group==gn).to_numpy()
    for quantity in ('R_CFU_ml','T1_CFU_ml','T2_CFU_ml',
                     'resistant_R_plus_T1_CFU_ml',
                     'R_fraction','T1_fraction','T2_fraction','resistant_fraction'):
        x=values[quantity]
        for aggregation,func in [('mean',lambda a:a.mean(axis=1)),
                                 ('median_across_animals',lambda a:np.median(a,axis=1))]:
            t=func(x[:,mt]);n=func(x[:,mn]); delta=t-n
            # ratio of group summaries, not a baseline-controlled percentage reduction
            ratio=t/n
            qd=quants(delta);qr=quants(ratio)
            comparisons.append(dict(endpoint=endpoint,quantity=quantity,
                aggregation=aggregation,
                diff_Target_minus_NonTarget_q025=qd[0],
                diff_Target_minus_NonTarget_median=qd[1],
                diff_Target_minus_NonTarget_q975=qd[2],
                target_over_nontarget_q025=qr[0],
                target_over_nontarget_median=qr[1],
                target_over_nontarget_q975=qr[2],
                posterior_P_Target_lower=float((delta<0).mean()),
                time_matched=endpoint.startswith('Early')))
contrast=pd.DataFrame(comparisons)
contrast.to_csv(RESULT_DIR/'target_nontarget_comparisons.csv',index=False)
display(contrast[(contrast.aggregation=='median_across_animals')].round(4))
normalized_contrast = contrast[(contrast.aggregation=='mean') &
                               contrast.quantity.isin(['R_fraction','T1_fraction','T2_fraction'])].copy()
normalized_contrast.to_csv(RESULT_DIR/'normalized_R_T1_T2_target_vs_nontarget.csv',index=False)
print('Target vs non-target contrasts for normalized components (group mean across animals):')
display(normalized_contrast.round(5))
print('Negative Target−Non-target difference means lower value in target;')
print('a ratio >1 means the target group has higher values.')
print('These comparisons do not demonstrate CRISPR-caused changes from baseline.')
if not valid:print('WARNING: convergence failed — contrasts are provisional.')


In [ ]:
#@title NB10.10 — Posterior predictive and external LB checks
log_rep = np.asarray(fit.stan_variable('log_y_rep'))
assert log_rep.shape[1:] == (len(data),3)
ppc_rows=[]
for i,rec in data.iterrows():
    for k,col in enumerate(COLS):
        obs=np.log(float(rec[col])); q=quants(log_rep[:,i,k])
        ppc_rows.append(dict(animal_id=rec.animal_id,group=rec.group,plate=col,
            observed_CFU_ml=float(rec[col]),observed_log_CFU_ml=obs,
            predictive_log_q025=q[0],predictive_log_median=q[1],
            predictive_log_q975=q[2],
            outside_predictive_95=bool(obs<q[0] or obs>q[2])))
ppc=pd.DataFrame(ppc_rows)
ppc.to_csv(RESULT_DIR/'selective_plate_PPC.csv',index=False)
print('In-sample PPC (not an independent test of accuracy):')
display(ppc.groupby('plate').agg(n=('outside_predictive_95','size'),
                                outside=('outside_predictive_95','sum')))
# LB was not fitted; this is a separate diagnostic of total viable burden.
lb_rows=[]
for i,rec in data[data.LB_CFU_ml.notna()].iterrows():
    lo,mid,hi=quants(all_burden[:,i]);obs=float(rec.LB_CFU_ml)
    lb_rows.append(dict(animal_id=rec.animal_id,group=rec.group,
        hours_post_infection=rec.hours_post_infection,observed_LB_CFU_ml=obs,
        latent_total_q025=lo,latent_total_median=mid,latent_total_q975=hi,
        observed_over_estimated_total=obs/mid,
        outside_latent_95=bool(obs<lo or obs>hi)))
lb_check=pd.DataFrame(lb_rows)
lb_check.to_csv(RESULT_DIR/'LB_external_check.csv',index=False)
print('Later LB observations outside latent total 95% intervals:',
      int(lb_check.outside_latent_95.sum()),'of',len(lb_check))
display(lb_check.round(2))
if ppc.outside_predictive_95.mean()>0.1:
    print('WARNING: In-sample posterior predictive undercoverage >10%; inspect errors.')


In [ ]:
#@title NB10.11 — Plot absolute resistant burden and resistant fraction
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for ax,qty,label in [
    (axes[0],'resistant_R_plus_T1_CFU_ml','R + T1 (CFU/mL)'),
    (axes[1],'resistant_fraction','(R + T1) / (R + T2)')]:
    sub=by_group[(by_group.aggregation=='median_across_animals') &
                 (by_group.quantity==qty)].set_index('group').loc[GROUPS]
    x=np.arange(len(sub));y=sub['median'].to_numpy();
    low=y-sub['q025'].to_numpy();high=sub['q975'].to_numpy()-y
    ax.errorbar(x,y,yerr=np.vstack([low,high]),fmt='o',capsize=4)
    ax.set_xticks(x,[s.replace(' — ','\n') for s in GROUPS])
    if qty.endswith('_CFU_ml'):ax.set_yscale('log')
    else:ax.set_ylim(0,1)
    ax.set_ylabel(label)
    ax.grid(axis='y',alpha=.25)
fig.suptitle('NB10 v1.2 — exploratory PE group medians (95% credible intervals)')
fig.tight_layout()
fig.savefig(RESULT_DIR/'NB10_v1_2_PE_resistant_population.png',dpi=180)
plt.show()


In [ ]:
#@title NB10.12 — Plot normalized R, T1 and T2 by target status
# Per-animal posterior fractions share the denominator R+T2.
# Each figure shows mean across the eight animals in each group.
import matplotlib.pyplot as plt
plot_groups=['Early — Target','Early — Non-target','Later — Target','Later — Non-target']
for quantity, ylabel in [('R_fraction','R / (R + T2)'),
                         ('T1_fraction','T1 / (R + T2)'),
                         ('T2_fraction','T2 / (R + T2)')]:
    plot_table=normalized_group[normalized_group.quantity==quantity].set_index('group').loc[plot_groups]
    xs=np.arange(4)
    ys=plot_table['median'].to_numpy()
    lower=ys-plot_table['q025'].to_numpy()
    upper=plot_table['q975'].to_numpy()-ys
    fig,ax=plt.subplots(figsize=(8,4.5))
    ax.errorbar(xs,ys,yerr=np.vstack((lower,upper)),fmt='o',capsize=4)
    ax.set_xticks(xs,[g.replace(' — ','\n') for g in plot_groups])
    ax.set_ylabel(ylabel)
    ax.set_ylim(0,1)
    ax.grid(axis='y',alpha=.25)
    ax.set_title(f'PE: normalized {quantity.replace("_fraction", "")} (posterior medians and 95% CrIs)')
    fig.tight_layout()
    fig.savefig(RESULT_DIR/f'NB10_v1_2_PE_{quantity}.png',dpi=180)
    plt.show()
print('R and T2 fractions sum to one; T1 is a subset of T2.')
print('Later target/non-target sampling times differ (16–20 h vs 24 h).')
if not valid:print('WARNING: Stan convergence failed — estimates are provisional.')


In [ ]:
#@title NB10.13 — Write analysis summary and interpretation safeguards
from datetime import datetime, timezone
summary_text=f'''NB10 v1.2 — PE exploratory R and total resistant burden model
Date UTC: {datetime.now(timezone.utc).isoformat()}
Animals: 32 independent PE animals (8 in each group)
Original PE source: NMS003_Part 6_ client data_CFUs (3).xlsx
SHA-256 of embedded source-derived CSV: {CSV_SHA}
Biology: PE has R, T1, T-sensitive; no donor bacteria
Spontaneous resistance plasmid loss in R: fixed at zero (assumption)
Plate meanings: Cb=R+T1; CbTc=T1; Tc=T1+T-sensitive
Primary outcomes: R, T1, T2, R+T1 absolute CFU/mL and normalized fractions
Fractions: R/(R+T2), T1/(R+T2), T2/(R+T2), (R+T1)/(R+T2)
Normalization per animal/posterior draw; T1 is a subset of T2
Normalized fraction summaries by group and target vs non-target contrasts saved separately
Measurement log SD (fixed, unvalidated): {PLATE_LOG_SD.tolist()}
Animal heterogeneity log SD (fixed, unvalidated): {ANIMAL_LOG_SD.tolist()}
Likelihood: Student-t(4) on natural-log CFU/mL; raw plate estimates not clipped
Convergence pass: {valid}; max R-hat: {max_rhat:.5f}; min ESS: {min_ess:.0f}
Divergences: {div}; treedepth hits: {tree_hits}
In-sample PPC outside 95%: {int(ppc.outside_predictive_95.sum())}/{len(ppc)}
Later LB outside latent total 95%: {int(lb_check.outside_latent_95.sum())}/{len(lb_check)}
6 h target/non-target comparison is time matched.
Later target 16–20 h vs non-target 24 h: NOT time matched.
No baseline measurements included: between-group burden comparisons are not
causal estimates of treatment-induced reduction.
Fixed measurement SD and biological heterogeneity require sensitivity analysis.
Posterior predictive coverage does not verify measurement or biological assumptions.
No changes to NB05, NB09, NB10 v1.0, manuscript, supplement or GitHub.
Private KWS-derived data: do not redistribute without permission.
'''
(RESULT_DIR/'NB10_v1_2_analysis_summary.txt').write_text(summary_text)
print(summary_text)


In [ ]:
#@title NB10.14 — Sensitivity check guidance (do not automatically refit)
# The model cannot identify true plate-error variance from three selective plates
# per animal. PLATE_LOG_SD and ANIMAL_LOG_SD are explicit assumptions.
# To test sensitivity, first preserve the current results, then run the notebook
# separately with these values in Cell NB10.05 and a distinct output folder:
#   Scenario A: PLATE_LOG_SD = [0.35, 0.35, 0.35]  (less noise)
#   Scenario B: PLATE_LOG_SD = [1.00, 1.00, 1.00]  (more noise)
# Keep ANIMAL_LOG_SD unchanged initially. Do not pool scenarios.
# Compare directions and 95% intervals for the 6 h R and R+T1 contrasts,
# posterior predictive checks, and the 16 later LB checks.
# No validation, publication or manuscript changes are implied by a PASS R-hat.
print('Sensitivity analysis is OPTIONAL and has not been run.')
print('First share the v1.2 NB10_PE_v1_2 results for convergence and model checks.')
